In [9]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

In [10]:
#Load Data
def extract_text(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read().splitlines()

train_text = extract_text('train_text.txt')
val_text = extract_text('val_text.txt')
test_text = extract_text('test_text.txt')

train_labels = pd.read_csv('train_labels.txt', header=None).values.ravel()
val_labels = pd.read_csv('val_labels.txt', header=None).values.ravel()
test_labels = pd.read_csv('test_labels.txt', header=None).values.ravel()

In [12]:
#Tokenization
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()


In [13]:
#Build Vocab
all_tokens = []
for text in train_text:
    all_tokens.extend(tokenize(text))

vocab = Counter(all_tokens)

# keep most common words (important for internship polish)
max_vocab_size = 20000

most_common = vocab.most_common(max_vocab_size - 2)

word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for i, (word, _) in enumerate(most_common, start=2):
    word2idx[word] = i

vocab_size = len(word2idx)

In [14]:
#Encoding
max_len = 50

def encode(text):
    tokens = tokenize(text)
    ids = [word2idx.get(t, 1) for t in tokens]
    return ids[:max_len] + [0] * (max_len - len(ids))

In [15]:
#Dataset Class
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = [encode(t) for t in texts]
        self.y = labels

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float)
        )

In [16]:
#DataLoaders
train_loader = DataLoader(TextDataset(train_text, train_labels), batch_size=32, shuffle=True)
val_loader = DataLoader(TextDataset(val_text, val_labels), batch_size=32)
test_loader = DataLoader(TextDataset(test_text, test_labels), batch_size=32)

In [17]:
#Embedding
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        emb = self.embedding(x)          # (batch, seq, embed)
        pooled = emb.mean(dim=1)         # simple averaging
        out = self.fc(pooled)
        return torch.sigmoid(out).squeeze()

In [18]:
#Training Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentimentModel(vocab_size).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [19]:
#Training Loop
def train_model(model, loader):
    model.train()
    total_loss = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(X)

        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [20]:
#Evaluation
def evaluate(model, loader):
    model.eval()
    preds_list, labels_list = [], []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            preds = model(X).cpu().numpy()

            preds = (preds > 0.5).astype(int)

            preds_list.extend(preds)
            labels_list.extend(y.numpy())

    acc = accuracy_score(labels_list, preds_list)
    f1 = f1_score(labels_list, preds_list, average="macro")

    return acc, f1

In [21]:
#Main Training Loop
epochs = 5

for epoch in range(epochs):
    loss = train_model(model, train_loader)
    val_acc, val_f1 = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    print("-" * 30)


RuntimeError: all elements of target should be between 0 and 1

In [ ]:
#Final Evaluation on Test Set
test_acc, test_f1 = evaluate(model, test_loader)

print("TEST RESULTS")
print("Accuracy:", test_acc)
print("F1 Score:", test_f1)